# BNP Paribas × Engie — pairs trading and CAC 40 screener

This notebook follows the BNP–Société Générale pipeline, replacing Société Générale with Engie. It also scans the CAC 40 universe for the 10 pairs with the smallest Engle–Granger p-values and plots them.

The screener uses the current CAC 40 constituents over historical prices, so the universe has survivorship bias. P-values are unadjusted for testing many pairs; they identify candidates for further research, not profitable trades.

In [ ]:
from pathlib import Path
import sys

# Supports running Jupyter from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import download_adjusted_closes
from src.analytics import kalman_hedge_ratio, dynamic_spread, rolling_zscore
from src.backtest import PairsBacktester
from src.screener import CAC40_TICKERS, fetch_universe_data, scan_cointegrated_pairs

## 1. Screen the CAC 40 universe

This runs the Engle–Granger test on all available pairs, then keeps the 10 smallest p-values. A small p-value is evidence of cointegration in this historical sample; it is not a forecast of returns.

In [ ]:
# This can take a little while because it downloads prices for the full universe.
universe_prices = fetch_universe_data(CAC40_TICKERS, start_date="2021-01-01")
cointegrated_pairs = scan_cointegrated_pairs(universe_prices)
top_10_pairs = cointegrated_pairs.head(10).copy()

if top_10_pairs.empty:
    print("No pairs passed the cointegration p-value threshold.")
else:
    display(top_10_pairs)

In [ ]:
if not top_10_pairs.empty:
    plot_pairs = top_10_pairs.sort_values("p_value", ascending=True).iloc[::-1]
    labels = [
        f"{row.ticker1} – {row.ticker2} (β={row.beta:.2f})"
        for row in plot_pairs.itertuples()
    ]
    evidence = -np.log10(plot_pairs["p_value"])

    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(labels, evidence, color="steelblue")
    ax.set_xlabel("−log₁₀(p-value)  (longer bar = smaller p-value)")
    ax.set_title("10 CAC 40 pairs with the smallest Engle–Granger p-values")
    ax.grid(axis="x", alpha=0.25)
    for bar, p_value in zip(bars, plot_pairs["p_value"]):
        ax.text(
            bar.get_width() + 0.03,
            bar.get_y() + bar.get_height() / 2,
            f"p={p_value:.3g}",
            va="center",
            fontsize=8,
        )
    fig.tight_layout()
    plt.show()

## 2. Analyze BNP Paribas × Engie

The pair analysis below uses the same Kalman hedge-ratio, rolling spread z-score, and backtest flow as the original BNP–Société Générale notebook.

In [ ]:
# yfinance treats the end date as exclusive. This includes available data through 2026.
PAIR_TICKERS = ("BNP.PA", "ENGI.PA")
prices = download_adjusted_closes(
    start="2020-01-01",
    end="2026-12-31",
    tickers=PAIR_TICKERS,
)
prices = prices.dropna().sort_index()
print(f"Rows: {len(prices):,}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail()

In [ ]:
# Dynamic hedge ratio and residual spread, estimated from adjusted price levels.
kalman_state = kalman_hedge_ratio(
    prices,
    dependent="BNP.PA",
    independent="ENGI.PA",
)
spread = dynamic_spread(\n    prices, kalman_state, dependent=\"BNP.PA\", independent=\"ENGI.PA\"\n)
zscore = rolling_zscore(spread, window=60)
beta_t = kalman_state["hedge_ratio"]

ENTRY_Z = 1.5
EXIT_Z = 0.0
backtester = PairsBacktester(
    spread_zscore=zscore,
    prices=prices[list(PAIR_TICKERS)],
    hedge_ratio=beta_t,
    entry_threshold=ENTRY_Z,
    exit_threshold=EXIT_Z,
    transaction_fee=0.001,  # 10 bps per unit of turnover
)
summary, pnl = backtester.run()
pd.Series(summary, name="Strategy summary").to_frame()

In [ ]:
# Equal-weight buy-and-hold benchmark: 50% in each adjusted-price series.
normalized_prices = prices / prices.iloc[0] * 100.0
buy_hold = (normalized_prices / 100.0).mean(axis=1) - 1.0
buy_hold = buy_hold.reindex(pnl.index).ffill()

positions = pnl["position"].reindex(zscore.index).fillna(0.0)
previous_positions = positions.shift(1).fillna(0.0)
long_entries = (positions == 1.0) & (previous_positions == 0.0)
short_entries = (positions == -1.0) & (previous_positions == 0.0)
exits = (positions == 0.0) & (previous_positions != 0.0)

fig, axes = plt.subplots(
    3, 1, figsize=(15, 14), sharex=True,
    gridspec_kw={"height_ratios": [1.1, 1.0, 1.6]},
)

# 1) Normalized adjusted prices.
axes[0].plot(normalized_prices.index, normalized_prices["BNP.PA"], label="BNP Paribas (BNP.PA)")
axes[0].plot(normalized_prices.index, normalized_prices["ENGI.PA"], label="Engie (ENGI.PA)")
axes[0].set_title("1. Normalized adjusted prices (start = 100)")
axes[0].set_ylabel("Index level")
axes[0].legend(loc="upper left")
axes[0].grid(alpha=0.25)

# 2) Dynamic Kalman hedge ratio.
axes[1].plot(beta_t.index, beta_t, color="darkorange", label="Dynamic hedge ratio βₜ")
axes[1].set_title("2. Kalman dynamic hedge ratio")
axes[1].set_ylabel("βₜ")
axes[1].legend(loc="upper left")
axes[1].grid(alpha=0.25)

# 3) Spread z-score, entry/exit markers, and cumulative performance.
ax_z = axes[2]
ax_z.plot(zscore.index, zscore, color="steelblue", label="Spread z-score")
ax_z.axhline(ENTRY_Z, color="firebrick", linestyle="--", label=f"Short-spread entry (+{ENTRY_Z})")
ax_z.axhline(-ENTRY_Z, color="seagreen", linestyle="--", label=f"Long-spread entry (−{ENTRY_Z})")
ax_z.axhline(EXIT_Z, color="gray", linestyle=":", label=f"Exit level ({EXIT_Z})")
ax_z.scatter(zscore.index[long_entries], zscore.loc[long_entries], marker="^", color="limegreen", s=45, label="Buy pair / long spread", zorder=4)
ax_z.scatter(zscore.index[short_entries], zscore.loc[short_entries], marker="v", color="tomato", s=45, label="Sell pair / short spread", zorder=4)
ax_z.scatter(zscore.index[exits], zscore.loc[exits], marker="x", color="black", s=40, label="Exit position", zorder=4)
ax_z.set_title("3. BNP–Engie z-score signals and cumulative PnL vs. buy and hold")
ax_z.set_ylabel("Spread z-score")
ax_z.grid(alpha=0.25)

ax_pnl = ax_z.twinx()
ax_pnl.plot(pnl.index, pnl["cumulative_pnl"] * 100.0, color="purple", linewidth=1.8, label="Pairs strategy net PnL")
ax_pnl.plot(buy_hold.index, buy_hold * 100.0, color="saddlebrown", linewidth=1.5, label="50/50 buy-and-hold")
ax_pnl.set_ylabel("Cumulative return (%)")

z_lines, z_labels = ax_z.get_legend_handles_labels()
p_lines, p_labels = ax_pnl.get_legend_handles_labels()
ax_z.legend(z_lines + p_lines, z_labels + p_labels, loc="upper left", ncol=2, fontsize=8)
axes[2].set_xlabel("Date")
fig.tight_layout()
plt.show()

## Interpretation notes

- The top-10 chart ranks statistical evidence in the full sample; it does not rank expected profitability.
- The scan tests many combinations, so some small p-values can occur by chance. Confirm candidates on a separate chronological period.
- The CAC 40 list reflects the current universe, not historical membership for each date.
- The 50/50 buy-and-hold benchmark is long both stocks and is not market-neutral.
- Backtest results depend on the sample period, rolling window, entry/exit levels, hedge ratio, and transaction-cost assumptions.